In [1]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Generate 100 mockup transaction records for a Regulatory Report
data = {
    'Transaction_ID': [f"TXN{1000 + i}" for i in range(100)],
    'Account_Number': [f"ACT{np.random.randint(10000, 99999)}" for i in range(100)],
    'Transaction_Type': np.random.choice(['Deposit', 'Withdrawal', 'Transfer', 'Fee'], 100),
    'Amount': np.round(np.random.uniform(10.0, 50000.0, 100), 2),
    'Currency': ['USD'] * 100,
    'Reported_Date': pd.date_range(start='2026-07-01', periods=100, freq='h').strftime('%Y-%m-%d').tolist()
}

df = pd.DataFrame(data)

# Inject intentional Data Quality issues (Exceptions)
df.loc[5, 'Amount'] = -2500.00         # Issue 1: Negative amount (Validity Error)
df.loc[12, 'Amount'] = np.nan          # Issue 2: Missing amount (Completeness Error)
df.loc[28, 'Account_Number'] = None    # Issue 3: Missing Account Number (Completeness Error)
df.loc[45, 'Currency'] = 'XYZ'         # Issue 4: Invalid Currency Code (Consistency Error)
df.loc[67, 'Reported_Date'] = '07/15/26' # Issue 5: Wrong Date Format (Validity Error)

# Save to a virtual CSV file to simulate data ingestion
df.to_csv('raw_regulatory_data.csv', index=False)
print("Phase 1 Complete: 'raw_regulatory_data.csv' successfully created with intentional anomalies!")

Phase 1 Complete: 'raw_regulatory_data.csv' successfully created with intentional anomalies!


In [2]:
# 1. Read the raw data file we created in Phase 1
df_raw = pd.read_csv('raw_regulatory_data.csv')

# 2. Create an empty list to store any data quality issues we find
exceptions = []

# 3. Loop through every single row to check our compliance rules
for index, row in df_raw.iterrows():
    txn_id = row['Transaction_ID']

    # --- Rule 1: Check for Completeness (Missing Values) ---
    if pd.isna(row['Account_Number']):
        exceptions.append({'Transaction_ID': txn_id, 'Field': 'Account_Number', 'Error_Type': 'Completeness', 'Details': 'Missing Account Number'})

    if pd.isna(row['Amount']):
        exceptions.append({'Transaction_ID': txn_id, 'Field': 'Amount', 'Error_Type': 'Completeness', 'Details': 'Missing Transaction Amount'})

    # --- Rule 2: Check for Validity (Logical Values) ---
    # Only check the amount if it isn't missing
    if not pd.isna(row['Amount']) and row['Amount'] <= 0:
        exceptions.append({'Transaction_ID': txn_id, 'Field': 'Amount', 'Error_Type': 'Validity', 'Details': f"Negative or zero amount found: {row['Amount']}"})

    # --- Rule 3: Check for Consistency (Standardized Formats) ---
    if row['Currency'] != 'USD':
        exceptions.append({'Transaction_ID': txn_id, 'Field': 'Currency', 'Error_Type': 'Consistency', 'Details': f"Unauthorized currency code: {row['Currency']}"})

# 4. Convert our exceptions list into a formal DataFrame report
df_exceptions = pd.DataFrame(exceptions)

# 5. Save the exceptions log to a separate CSV file for stakeholders
df_exceptions.to_csv('data_quality_exceptions.csv', index=False)

print(f"Phase 2 Complete: Scanned 100 records and successfully isolated {len(df_exceptions)} data quality violations!")
print("\n--- Snippet of Isolated Exceptions ---")
print(df_exceptions.head())

Phase 2 Complete: Scanned 100 records and successfully isolated 4 data quality violations!

--- Snippet of Isolated Exceptions ---
  Transaction_ID           Field    Error_Type  \
0        TXN1005          Amount      Validity   
1        TXN1012          Amount  Completeness   
2        TXN1028  Account_Number  Completeness   
3        TXN1045        Currency   Consistency   

                                  Details  
0  Negative or zero amount found: -2500.0  
1              Missing Transaction Amount  
2                  Missing Account Number  
3         Unauthorized currency code: XYZ  


In [3]:
# 1. Load the total records and total exceptions
total_records_evaluated = len(df_raw)
total_exceptions_found = len(df_exceptions)

# 2. Calculate the core KPI: Data Health Score (%)
# Formula: (Clean Records / Total Records) * 100
clean_records = total_records_evaluated - total_exceptions_found
data_health_score = (clean_records / total_records_evaluated) * 100

# 3. Aggregate exceptions by Error Type to show where the main systemic issues lie
error_summary = df_exceptions['Error_Type'].value_counts()

# 4. Print the Executive Data Quality Summary Report
print("==================================================")
print("     ENTERPRISE DATA QUALITY HEALTH REPORT        ")
print("==================================================")
print(f"Total Records Evaluated : {total_records_evaluated}")
print(f"Total Exceptions Found  : {total_exceptions_found}")
print(f"Overall Data Health Score: {data_health_score:.1f}%")
print("--------------------------------------------------")
print("Breakdown of Risk Events by Error Type:")
for error_type, count in error_summary.items():
    print(f" - {error_type}: {count} instance(s)")
print("==================================================")
print("Status: 'data_quality_exceptions.csv' exported for stakeholder remediation workflows.")

     ENTERPRISE DATA QUALITY HEALTH REPORT        
Total Records Evaluated : 100
Total Exceptions Found  : 4
Overall Data Health Score: 96.0%
--------------------------------------------------
Breakdown of Risk Events by Error Type:
 - Completeness: 2 instance(s)
 - Validity: 1 instance(s)
 - Consistency: 1 instance(s)
Status: 'data_quality_exceptions.csv' exported for stakeholder remediation workflows.
